# Visualise More Reconstructions and Generations

By default, 40 example generations are recorded during the test run.

This is a simple notebook that allows you to view more generations, as well as compare reconstructions in more detail.

## Preliminaries

The repository assumes all entry points to be run as modules from root. Since
this notebook is 2 levels deep, the below block moves us to the root directory.

In [ ]:
import os

# Change this variable if the root folder name has been changed
root_dir = "nvae-shape-encoding"
current_dir = os.getcwd()

if not current_dir.endswith(root_dir):
    %cd ../..

assert os.getcwd().endswith(root_dir)

Update `model_path` and `model_path_vae` to choose the models. We will be comparing the reconstruction quality between the NVAE and VAE models.

In [ ]:
import lightning as L
import torch

from arch.nvae.nvae import NVAE
from arch.vae.vae import VAE
from utils.const import SEED
from data_modules.acdc import ACDCMaskDataModule
from utils.utils import setup_device

model_path = "logs/nvae_acdc/best/default/checkpoints/epoch=97-step=20972.ckpt"
model_path_vae = "logs/vae_acdc/best/beta-vae/checkpoints/epoch=47-step=5136.ckpt"

# Setup device
device = setup_device()
print(f"Device: {device}")

# Seed
L.seed_everything(SEED)

# Load data
data_module = ACDCMaskDataModule(batch_size=20)

# Reseed after preprocessing data
L.seed_everything(SEED)

# Load model
model = NVAE.load_from_checkpoint(model_path)
model_vae = VAE.load_from_checkpoint(model_path_vae)

In [ ]:
from utils.utils import get_data

loader_test = data_module.test_dataloader(shuffle=True)
feats_all = get_data(loader_test)
feats_all.shape

## Compare Reconstructions

Set up to view reconstructions.

In [ ]:
from utils.eval import get_samples_and_reconstructions_pixel_diff
from utils.utils import show_samples

with torch.no_grad():
    model.eval()
    model.to(device)
    feats_all = feats_all.to(device)
    
    num_data = feats_all.shape[0]
    samples_idx = torch.randperm(num_data)[:6]
    feats = feats_all[samples_idx]
    feats_hat_logits, _, _, _, _ = model(feats, test=True)
    
with torch.no_grad():
    model_vae.eval()
    model_vae.to(device)
    _, _, _, x_hat_logits_vae = model_vae(feats, test=True)

samples, reconstructions, reconstruction_pixel_error = get_samples_and_reconstructions_pixel_diff(
    feats,
    feats_hat_logits,
    return_reconstructions=True,
)

_, reconstructions_vae, reconstruction_pixel_error_vae = get_samples_and_reconstructions_pixel_diff(
    feats,
    x_hat_logits_vae,
    return_reconstructions=True,
)

View and compare reconstructions.

In [ ]:
import matplotlib.pyplot as plt
from torchvision.utils import make_grid

from utils.colourmap import GIRIDIS, GREDS

fig, ax = plt.subplots(1, 5, figsize=(30, 36))

# Column 1: Ground truth

samples = samples.cpu().float()
samples_grid = make_grid(samples, nrow=1, padding=2, normalize=True)
samples_grid = samples_grid[0]

ax[0].axis("off")
ax[0].imshow(samples_grid, cmap=GIRIDIS)
ax[0].set_title("Ground Truth", fontsize=36, pad=36)

# Column 2: NVAE reconstruction

reconstructions = reconstructions.cpu().float()
reconstructions_grid = make_grid(reconstructions, nrow=1, padding=2, normalize=True)
reconstructions_grid = reconstructions_grid[0]

ax[1].axis("off")
ax[1].imshow(reconstructions_grid, cmap=GIRIDIS)
ax[1].set_title("NVAE\nReconstruction", fontsize=36, pad=36)

# Column 3: VAE reconstruction

reconstructions_vae = reconstructions_vae.cpu().float()
reconstructions_vae_grid = make_grid(reconstructions_vae, nrow=1, padding=2, normalize=True)
reconstructions_vae_grid = reconstructions_vae_grid[0]

ax[2].axis("off")
ax[2].imshow(reconstructions_vae_grid, cmap=GIRIDIS)
ax[2].set_title("VAE\nReconstruction", fontsize=36, pad=36)

# Column 4: NVAE reconstruction error

reconstruction_pixel_error = reconstruction_pixel_error.cpu().float()
reconstruction_pixel_error_grid = make_grid(reconstruction_pixel_error, nrow=1, padding=2, normalize=True)
reconstruction_pixel_error_grid = reconstruction_pixel_error_grid > 0
reconstruction_pixel_error_grid = reconstruction_pixel_error_grid[0]

ax[3].axis("off")
ax[3].imshow(reconstruction_pixel_error_grid, cmap=GREDS)
ax[3].set_title("NVAE\nIncorrect Pixels", fontsize=36, pad=36)

# Column 5: VAE reconstruction error

reconstruction_pixel_error_vae = reconstruction_pixel_error_vae.cpu().float()
reconstruction_pixel_error_vae_grid = make_grid(reconstruction_pixel_error_vae, nrow=1, padding=2, normalize=True)
reconstruction_pixel_error_vae_grid = reconstruction_pixel_error_vae_grid > 0
reconstruction_pixel_error_vae_grid = reconstruction_pixel_error_vae_grid[0]

ax[4].axis("off")
ax[4].imshow(reconstruction_pixel_error_vae_grid, cmap=GREDS)
ax[4].set_title("VAE\nIncorrect Pixels", fontsize=36, pad=36)

## Generations

Change `num_samples` on line 1 to view more generations. To have a good figure, you may also want to change line 12: `ncol` is number of columns, `figsize` is figure size.

In [ ]:
num_samples = 24

with torch.no_grad():
    model.eval()
    model.to(device)
    
    x_fake = model.decoder.generate(num_samples=num_samples, device=device)
    feats_fake = model.conditional_coder(x_fake)

# Discretise probabilistic map then view generations
generations = torch.argmax(feats_fake, dim=1).unsqueeze(1)
show_samples(generations, rgb=False, ncol=6, figsize=(24, 16))